# Arabic-to-English Speech Translation — Inference Demo

This notebook runs the fine-tuned SeamlessM4T-v2 + LoRA model from Hugging Face.
Upload any Arabic audio file or use the provided sample to get English text and speech.

**Pipeline:**


**Model:** [hams-chadi/seamless-m4t-v2-arabic-english-lora](https://huggingface.co/hams-chadi/seamless-m4t-v2-arabic-english-lora)


## 1. Setup

In [ ]:
import time
import json
import numpy as np
import torch
import librosa
import soundfile as sf
from pathlib import Path
from IPython.display import Audio as IPyAudio, display
from transformers import AutoProcessor, SeamlessM4Tv2ForSpeechToText
from peft import PeftModel

HF_MODEL_ID    = "hams-chadi/seamless-m4t-v2-arabic-english-lora"
BASE_MODEL_ID  = "facebook/seamless-m4t-v2-large"
TGT_LANG       = "eng"
SAMPLE_RATE    = 16_000
MAX_NEW_TOKENS = 128
NUM_BEAMS      = 5
DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


## 2. Load Model from Hugging Face

In [ ]:
print("Loading processor ...")
processor = AutoProcessor.from_pretrained(HF_MODEL_ID)

print("Loading base model ...")
base = SeamlessM4Tv2ForSpeechToText.from_pretrained(
    BASE_MODEL_ID, dtype=torch.float32
)

print("Loading LoRA adapter ...")
peft_model = PeftModel.from_pretrained(base, HF_MODEL_ID)
model = peft_model.merge_and_unload().to(DEVICE).eval()

print("Model ready.")


## 3. Load Kokoro TTS

In [ ]:
from kokoro import KPipeline
import soundfile as sf

kokoro = KPipeline(lang_code="a", device=DEVICE)
TTS_SR = 24000
print("Kokoro TTS ready.")

def synthesize(text, output_path="output.wav"):
    text = (text or "").strip()
    if not text:
        return np.zeros(TTS_SR, dtype=np.float32), TTS_SR
    chunks = []
    for _, _, audio in kokoro(text, voice="af_heart", speed=1.0):
        if audio is not None and len(audio) > 0:
            chunks.append(np.asarray(audio, dtype=np.float32))
    wav = np.concatenate(chunks) if chunks else np.zeros(TTS_SR, dtype=np.float32)
    sf.write(output_path, wav, TTS_SR)
    return wav, TTS_SR


## 4. Run Inference

Set  to your Arabic audio file.
You can use any WAV or MP3 file, or one of the sample files in .


In [ ]:
# Set this to your Arabic audio file
AUDIO_PATH_1 = r"sample_audio\example_ar_short.wav"
AUDIO_PATH_2 = r"sample_audio\example_ar_medium.wav"
AUDIO_PATH_3 = r"sample_audio\example_ar_long.wav"

# Load audio

audio, _ = librosa.load(AUDIO_PATH_1, sr=SAMPLE_RATE, mono=True)
audio = audio.astype(np.float32)
duration_s = len(audio) / SAMPLE_RATE
print(f"Audio duration: {duration_s:.2f}s")

print("Playing Arabic input:")
display(IPyAudio(audio, rate=SAMPLE_RATE))


In [ ]:
# Step 1: Translate Arabic speech to English text
t0 = time.perf_counter()
inputs = processor(audios=audio, sampling_rate=SAMPLE_RATE, return_tensors="pt").to(DEVICE)
with torch.no_grad():
    tokens = model.generate(
        **inputs, tgt_lang=TGT_LANG,
        num_beams=NUM_BEAMS, max_new_tokens=MAX_NEW_TOKENS,
    )
text = processor.batch_decode(tokens, skip_special_tokens=True)[0]
t_s2tt = (time.perf_counter() - t0) * 1000

print(f"English text : {repr(text)}")
print(f"S2TT latency : {t_s2tt:.1f} ms")


In [ ]:
# Step 2: Convert English text to English speech
t0 = time.perf_counter()
wav, tts_sr = synthesize(text, "output.wav")
t_tts = (time.perf_counter() - t0) * 1000

total = t_s2tt + t_tts
rtf   = (total / 1000) / duration_s

print("Playing English output:")
display(IPyAudio(wav, rate=tts_sr))

result = {
    "source_audio"    : AUDIO_PATH,
    "translated_text" : text,
    "output_audio"    : "output.wav",
    "latency_ms": {
        "time_to_first_text" : round(t_s2tt, 1),
        "time_to_first_audio": round(total, 1),
        "total_end_to_end"   : round(total, 1),
    },
    "audio_duration_s" : round(duration_s, 2),
    "real_time_factor"  : round(rtf, 3),
}

print()
print(json.dumps(result, indent=2))
